# Data Preparation

In [15]:
import os, re, warnings, hashlib, unicodedata
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

# Utilitaires dispos (tu les as déjà importés plus haut)
try:
    from ftfy import fix_text
except Exception:
    def fix_text(x): return x

try:
    from langdetect import detect, LangDetectException
except Exception:
    detect, LangDetectException = None, Exception

from simhash import Simhash

RNG = 42
np.random.seed(RNG)

In [16]:
# Vérifs & normalisation colonnes
expected_cols = ["title", "text", "subject", "date", "label"]
assert all(c in df.columns for c in expected_cols), f"Colonnes manquantes: {set(expected_cols)-set(df.columns)}"

# Trim espaces, convertir en str où pertinent
for col in ["title", "text", "subject", "date"]:
    df[col] = df[col].astype(str).str.strip()

In [17]:
# 2) Nettoyage texte de base 
URL_RE = r"(https?://\S+|www\.\S+)"
HANDLE_RE = r"@\w+"
HTML_RE = r"<[^>]+>"
MULTI_WS_RE = r"\s+"

# mots à retirer souvent vus comme boilerplate
BOILERPLATE_PREFIXES = [
    r"^\(?reuters\)?\s*-\s*",      # "(Reuters) - " ou "Reuters - "
    r"^\(?afp\)?\s*-\s*",
]
boilerplate_prefix = re.compile("|".join(BOILERPLATE_PREFIXES), flags=re.IGNORECASE)

def normalize_text(x: str) -> str:
    # fix encodage & accents bizarres
    x = fix_text(x)
    # unicode normalize
    x = unicodedata.normalize("NFKC", x)
    # enlever html
    x = re.sub(HTML_RE, " ", x)
    # urls & handles
    x = re.sub(URL_RE, " ", x)
    x = re.sub(HANDLE_RE, " ", x)
    # emojis/symboles divers -> garder lettres/chiffres ponctuation légère
    x = re.sub(r"[^\w\s\.\,\!\?\:\;\%\-\$\'\"]+", " ", x)
    # retirer boilerplate en début
    x = boilerplate_prefix.sub("", x)
    # espaces
    x = re.sub(MULTI_WS_RE, " ", x).strip()
    # lowercase
    x = x.lower()
    return x

# Appliquer au titre & texte
df["title_clean"] = df["title"].map(normalize_text)
df["text_clean"]  = df["text"].map(normalize_text)

#  si tu veux une version ultra-sobre pour bag-of-words:
def strip_to_words(x: str) -> str:
    x = re.sub(r"[^a-zA-Z\s]", " ", x)
    x = re.sub(MULTI_WS_RE, " ", x).strip().lower()
    return x

df["title_alpha"] = df["title_clean"].map(strip_to_words)
df["text_alpha"]  = df["text_clean"].map(strip_to_words)

print("Exemple après nettoyage:")
display(df[["title", "title_clean"]].head(2))
display(df.loc[:, ["text"]].head(1))


Exemple après nettoyage:


,title,title_clean
0,Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing,donald trump sends out embarrassing new year s eve message; this is disturbing
1,Drunk Bragging Trump Staffer Started Russian Collusion Investigation,drunk bragging trump staffer started russian collusion investigation


,text
0,"Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and the very dishonest fake new..."


In [18]:
#3) Vides & lignes trop courtes ---
before = len(df)
mask_text_empty = df["text_clean"].str.len() < 10  # seuil minimal (ajuste si tu veux)
print(f"Lignes avec texte < 10 char: {mask_text_empty.sum()} / {before}")
df = df.loc[~mask_text_empty].copy()


Lignes avec texte < 10 char: 738 / 44898


In [19]:
# 4) Déduplication EXACTE (MD5) + contradictions (info)
def md5_pair(title_clean, text_clean):
    s = (title_clean + " || " + text_clean).encode("utf-8", "ignore")
    return hashlib.md5(s).hexdigest()

df["md5"] = [md5_pair(t, x) for t, x in zip(df["title_clean"], df["text_clean"])]

# Contradictions (mêmes contenus, labels différents)
dup_stats = df.groupby("md5")["label"].nunique()
n_contra = int((dup_stats > 1).sum())
print(f"⚠️ Contradictions de label (mêmes contenus): {n_contra}")

# Drop doublons exacts (garder 1er)
dup_exact_mask = df["md5"].duplicated(keep="first")
print(f"Doublons exacts retirés: {int(dup_exact_mask.sum())}")
df = df.loc[~dup_exact_mask].reset_index(drop=True)

⚠️ Contradictions de label (mêmes contenus): 0
Doublons exacts retirés: 5838


In [20]:
#5) Quasi-doublons (SimHash) — Hamming ≤ 3
def simhash_val(row):
    toks = re.findall(r"\w+", (row["title_alpha"] + " " + row["text_alpha"]))
    shingles = [" ".join(toks[i:i+3]) for i in range(len(toks)-2)] or toks
    return Simhash(shingles).value

df["simhash"] = df.apply(simhash_val, axis=1)
df["bucket"]  = df["simhash"].apply(lambda v: v >> (64 - 16))  # 16 bits de tête

def hamming(a, b): return (a ^ b).bit_count()

keep = np.ones(len(df), dtype=bool)
for b, grp in df.sort_values("bucket").groupby("bucket"):
    sims = grp["simhash"].values
    idxs = grp.index.values
    kept_local = []
    for i, idx_i in enumerate(idxs):
        if any(hamming(sims[i], sims[j]) <= 3 for j in kept_local):
            keep[idx_i] = False
        else:
            kept_local.append(i)

n_near = int((~keep).sum())
df = df.loc[keep].reset_index(drop=True)
df.drop(columns=["bucket"], inplace=True)
print(f"Quasi-doublons retirés (dist≤3): {n_near} | total={len(df)}")

# group_id pour splits sans fuite (recalcule propre)
df["group_id"] = [md5_pair(t, x) for t, x in zip(df["title_clean"], df["text_clean"])]


Quasi-doublons retirés (dist≤3): 25 | total=38297


In [21]:
# 6) Détection de langue
if detect is not None:
    def is_en(s):
        try: return detect(s) == "en"
        except LangDetectException: return False
    # estimation échantillon
    p_en = df["text_clean"].sample(min(3000, len(df)), random_state=RNG).map(is_en).mean()
    print(f"🌐 Estimation % anglais (échantillon): {100*p_en:.1f}%")
    df["is_en"] = df["text_clean"].map(is_en)
    removed = int((~df["is_en"]).sum())
    df = df.loc[df["is_en"]].drop(columns=["is_en"]).reset_index(drop=True)
    print(f"Non-EN retirés: {removed} | total={len(df)}")
else:
    print("LangDetect indisponible — pas de filtre langue.")



🌐 Estimation % anglais (échantillon): 99.9%
Non-EN retirés: 13 | total=38284


In [22]:
#  7) Dates: parsing + features calendrier 
def parse_date_safe(x: str):
    for fmt in ("%B %d, %Y", "%b %d, %Y"):
        try: return pd.to_datetime(x, format=fmt)
        except Exception: pass
    return pd.to_datetime(x, errors="coerce")

df["date_dt"] = df["date"].map(parse_date_safe)
n_bad = int(df["date_dt"].isna().sum())
print("Dates non parsées:", n_bad)
if n_bad > 0:
    df.loc[df["date_dt"].isna(), "date_dt"] = df["date_dt"].median()

df["year"]  = df["date_dt"].dt.year.astype(int)
df["month"] = df["date_dt"].dt.month.astype(int)
df["dow"]   = df["date_dt"].dt.dayofweek.astype(int)

Dates non parsées: 1


In [23]:
#  8) Normaliser subject + feature texte finale
df["subject"] = (df["subject"].str.strip()
                               .str.replace("_","-", regex=False)
                               .str.replace(" ","-", regex=False)
                               .str.replace(r"[-]{2,}","-", regex=True)
                               .str.lower())
SUBJ_MAP = {"politicsnews":"politics-news","worldnews":"world-news","us_news":"us-news"}
df["subject"] = df["subject"].map(lambda s: SUBJ_MAP.get(s, s))


In [24]:
#  9) Feature texte finale 
df["text_full"] = (df["title_clean"].fillna("") + " . " + df["text_clean"].fillna("")).str.strip()


In [25]:
# --- Split group-aware 80/10/10 + exports ---
import numpy as np, os, json, hashlib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

# group_id stable basé sur le contenu nettoyé
df["group_id"] = (df["title_clean"] + " || " + df["text_clean"]).map(
    lambda x: hashlib.md5(x.encode("utf-8","ignore")).hexdigest()
)

X_idx  = np.arange(len(df))
y      = df["label"].values
groups = df["group_id"].values

# 80 / 20
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(gss.split(X_idx, y, groups=groups))

# 10 / 10 (remap des indices relatifs → globaux)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_rel, test_rel = next(gss2.split(X_idx[temp_idx], y[temp_idx], groups=groups[temp_idx]))
val_idx  = temp_idx[val_rel]
test_idx = temp_idx[test_rel]

# sanity checks sur indices GLOBAUX
S_tr, S_va, S_te = set(train_idx), set(val_idx), set(test_idx)
assert S_tr.isdisjoint(S_va) and S_tr.isdisjoint(S_te) and S_va.isdisjoint(S_te)
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    labs = set(df.loc[idx, "label"].unique().tolist())
    assert labs == {0,1}, f"{name}: classes manquantes {labs}"
print("✅ Splits OK.")

# build DFs
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

def dist(name, d):
    vc = d["label"].value_counts(normalize=True).sort_index()
    return f"{name}: n={len(d)} | p(True=0)={vc.get(0,0):.3f} | p(Fake=1)={vc.get(1,0):.3f}"
print(dist("train", train_df))
print(dist("val",   val_df))
print(dist("test",  test_df))

# exports
os.makedirs("processed", exist_ok=True)
cols_keep = ["title","text","subject","date_dt","label","text_full","year","month","dow"]
train_df[cols_keep].to_csv("processed/train.csv", index=False)
val_df[cols_keep].to_csv("processed/val.csv", index=False)
test_df[cols_keep].to_csv("processed/test.csv", index=False)

cls_w = compute_class_weight("balanced", classes=np.array([0,1]), y=train_df["label"].values)
class_weights = {0: float(cls_w[0]), 1: float(cls_w[1])}
with open("processed/class_weights.json","w") as f: json.dump(class_weights, f, indent=2)

print("💾 Écrit dans ./processed  |  class_weights:", class_weights)


✅ Splits OK.
train: n=30627 | p(True=0)=0.547 | p(Fake=1)=0.453
val: n=3828 | p(True=0)=0.536 | p(Fake=1)=0.464
test: n=3829 | p(True=0)=0.551 | p(Fake=1)=0.449
💾 Écrit dans ./processed  |  class_weights: {0: 0.9141842278072951, 1: 1.1035961372153358}
